In [1]:
import pandas as pd
import numpy as np

# DATASET DESCRIPTION 

In [2]:
# Load dataset (adjust filename if needed)
df = pd.read_csv("Nike_Sales_Uncleaned.csv")
print("sucessfully read")

sucessfully read


In [3]:
print("First 5 rows:")
display(df.head())

First 5 rows:


,Order_ID,Gender_Category,Product_Line,Product_Name,Size,Units_Sold,MRP,Discount_Applied,Revenue,Order_Date,Sales_Channel,Region,Profit
0,2000,Kids,Training,SuperRep Go,M,NaN,NaN,0.47,0.0,2024-03-09,Online,bengaluru,-770.45
1,2001,Women,Soccer,Tiempo Legend,M,3.0,4957.93,NaN,0.0,2024-07-09,Retail,Hyd,-112.53
2,2002,Women,Soccer,Premier III,M,4.0,NaN,NaN,0.0,NaN,Retail,Mumbai,3337.34
3,2003,Kids,Lifestyle,Blazer Mid,L,NaN,9673.57,NaN,0.0,04-10-2024,Online,Pune,3376.85
4,2004,Kids,Running,React Infinity,XL,NaN,NaN,NaN,0.0,2024/09/12,Retail,Delhi,187.89


In [4]:
print("\nDataset Info:")
df.info()


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          2500 non-null   int64  
 1   Gender_Category   2500 non-null   object 
 2   Product_Line      2500 non-null   object 
 3   Product_Name      2500 non-null   object 
 4   Size              1990 non-null   object 
 5   Units_Sold        1265 non-null   float64
 6   MRP               1246 non-null   float64
 7   Discount_Applied  832 non-null    float64
 8   Revenue           2500 non-null   float64
 9   Order_Date        1884 non-null   object 
 10  Sales_Channel     2500 non-null   object 
 11  Region            2500 non-null   object 
 12  Profit            2500 non-null   float64
dtypes: float64(5), int64(1), object(7)
memory usage: 254.0+ KB


In [5]:
print("\nColumn Names:")
print(df.columns.tolist())


Column Names:
['Order_ID', 'Gender_Category', 'Product_Line', 'Product_Name', 'Size', 'Units_Sold', 'MRP', 'Discount_Applied', 'Revenue', 'Order_Date', 'Sales_Channel', 'Region', 'Profit']


# DATASET CLEANING 

In [8]:
#Inspect
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:\n", df.isnull().sum())
print("Unique values in key columns:")
print("Gender:", df['Gender_Category'].unique())
print("Product Line:", df['Product_Line'].unique())
print("Sales Channel:", df['Sales_Channel'].unique())
print("Region sample:", df['Region'].unique()[:10])

Shape: (2500, 13)
Columns: ['Order_ID', 'Gender_Category', 'Product_Line', 'Product_Name', 'Size', 'Units_Sold', 'MRP', 'Discount_Applied', 'Revenue', 'Order_Date', 'Sales_Channel', 'Region', 'Profit']
Missing values:
 Order_ID               0
Gender_Category        0
Product_Line           0
Product_Name           0
Size                 510
Units_Sold          1235
MRP                 1254
Discount_Applied    1668
Revenue                0
Order_Date           616
Sales_Channel          0
Region                 0
Profit                 0
dtype: int64
Unique values in key columns:
Gender: ['Kids' 'Women' 'Men']
Product Line: ['Training' 'Soccer' 'Lifestyle' 'Running' 'Basketball']
Sales Channel: ['Online' 'Retail']
Region sample: ['bengaluru' 'Hyd' 'Mumbai' 'Pune' 'Delhi' 'Bangalore' 'Hyderabad'
 'hyderbad' 'Kolkata']


In [9]:
# Drop duplicate Order_IDs
df = df.drop_duplicates(subset=['Order_ID'])

In [10]:
# Gender, Sales Channel, Product Line
df['Gender_Category'] = df['Gender_Category'].str.strip().str.title()
df['Sales_Channel'] = df['Sales_Channel'].str.strip().str.title()
df['Product_Line'] = df['Product_Line'].str.strip().str.title()

# Region – fix typos
region_map = {
    "bengaluru":"Bangalore", "Banglore":"Bangalore",
    "Hyd":"Hyderabad", "hyderbad":"Hyderabad"
}
df['Region'] = df['Region'].replace(region_map).str.title()

In [11]:
# Fill missing with "Unknown"
df['Size'] = df['Size'].fillna("Unknown").str.strip()

In [12]:
# Fill missing with 0, drop negatives
df['Units_Sold'] = df['Units_Sold'].fillna(0)
df = df[df['Units_Sold'] >= 0]

In [13]:
# Step: Fill missing MRP using same Product_Name values

# Create a mapping of Product_Name → most common MRP
mrp_map = df.groupby('Product_Name')['MRP'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

# Fill missing MRP using this mapping
df['MRP'] = df.apply(
    lambda row: mrp_map[row['Product_Name']] if pd.isna(row['MRP']) else row['MRP'],
    axis=1)

# If still missing, fill with median as fallback
df['MRP'] = df['MRP'].fillna(df['MRP'].median())

In [14]:
df['Revenue_Calc'] = df['Units_Sold'] * df['MRP'] * (1 - df['Discount_Applied']/100)

In [15]:
df['Revenue_Calc'] = df['Units_Sold'] * df['MRP'] * (1 - df['Discount_Applied']/100)

In [17]:
# Ensure all inputs are filled
df['Units_Sold'] = df['Units_Sold'].fillna(0)
df['MRP'] = df['MRP'].fillna(df['MRP'].median())
df['Discount_Applied'] = df['Discount_Applied'].fillna(0)

# Recalculate revenue safely
df['Revenue_Calc'] = df['Units_Sold'] * df['MRP'] * (1 - df['Discount_Applied']/100)

In [19]:
# validation
print("Missing values after cleaning:\n", df.isnull().sum())
print("Revenue stats:\n", df['Revenue_Calc'].describe())
print("Profit stats:\n", df['Profit_Recalc'].describe())

Missing values after cleaning:
 Order_ID               0
Gender_Category        0
Product_Line           0
Product_Name           0
Size                   0
Units_Sold             0
MRP                    0
Discount_Applied       0
Revenue                0
Order_Date           536
Sales_Channel          0
Region                 0
Profit                 0
Revenue_Calc           0
Profit_Recalc       1469
dtype: int64
Revenue stats:
 count     2194.000000
mean      3637.027347
std       6812.496951
min          0.000000
25%          0.000000
50%          0.000000
75%       5111.824780
max      39984.880000
Name: Revenue_Calc, dtype: float64
Profit stats:
 count      725.000000
mean      1103.229633
std       1990.398206
min          0.000000
25%          0.000000
50%          0.000000
75%       1668.592582
max      11386.697632
Name: Profit_Recalc, dtype: float64


In [20]:
# 1. Handle missing Order_Date
#Drop rows without Order_Date
df = df.dropna(subset=['Order_Date'])

# 2. Fix Profit_Recalc
df['Profit_Recalc'] = df['Profit_Recalc'].fillna(0)

In [21]:
print("Missing values after cleaning:\n", df.isnull().sum())
print("Revenue stats:\n", df['Revenue_Calc'].describe())
print("Profit stats:\n", df['Profit_Recalc'].describe())

Missing values after cleaning:
 Order_ID            0
Gender_Category     0
Product_Line        0
Product_Name        0
Size                0
Units_Sold          0
MRP                 0
Discount_Applied    0
Revenue             0
Order_Date          0
Sales_Channel       0
Region              0
Profit              0
Revenue_Calc        0
Profit_Recalc       0
dtype: int64
Revenue stats:
 count     1658.000000
mean      3678.152774
std       6740.983799
min          0.000000
25%          0.000000
50%          0.000000
75%       5545.161460
max      39984.880000
Name: Revenue_Calc, dtype: float64
Profit stats:
 count    1658.000000
mean      386.928876
std      1270.328149
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max      9876.192115
Name: Profit_Recalc, dtype: float64


In [22]:
df.to_csv("Nike_Sales_cleaned.csv", index=False)